# PCA + Malicious (D)DoS Flow Integration — CIC17 to CIC18

This notebook evaluates the integration of malicious **(D)DoS** network flows from **GenIDS-CIC17** into **GenIDS-CIC18** with dimensionality reduction using **Principal Component Analysis (PCA)**.

The procedure follows the same flow-integration methodology used in the baseline notebooks:

1. A percentage of malicious **(D)DoS** flows is selected from the external dataset.
2. The selected flows are integrated into the training dataset.
3. The integrated flows are removed from the external evaluation dataset to prevent data leakage.
4. An equivalent number of flows from the same class is removed from the training dataset before integration to preserve dataset size and reduce distribution distortion.
5. PCA is applied after scaling and before model training.

The default configuration uses a **60% malicious flow integration rate**, following the original notebook. Change `INTEGRATION_RATE` to `0.20`, `0.40`, `0.60`, or `0.80` to run the remaining scenarios.

## 1. Environment Setup

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    auc,
)

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

## 2. Experiment Configuration

Adjust the paths according to your local environment or repository structure.

In the original notebook, after label encoding, the **(D)DoS** class corresponds to class value `2`.

In [ ]:
RANDOM_STATE = 42
INTEGRATION_RATE = 0.60  # Change to 0.20, 0.40, 0.60, or 0.80 as needed.
TEST_SIZE = 0.80
PCA_COMPONENTS = 25

DATA_DIR = Path("../datasets")
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CIC17_PATH = DATA_DIR / "GenIDS-CIC17.csv"
CIC18_PATH = DATA_DIR / "GenIDS-CIC18.csv"

LABEL_COLUMN = "multiclass"
TIME_COLUMN = "bidirectional_first_seen_ms"
SOURCE_COLUMN = "source_dataset"
DDOS_CLASS_VALUE = 2

print(f"Integration rate: {INTEGRATION_RATE:.0%}")
print(f"PCA components: {PCA_COMPONENTS}")
print(f"(D)DoS class value: {DDOS_CLASS_VALUE}")

## 3. Helper Functions

In [ ]:
def load_dataset(path: Path, dataset_name: str) -> pd.DataFrame:
    """Load a dataset from a CSV file."""
    if not path.exists():
        raise FileNotFoundError(
            f"File not found for {dataset_name}: {path}. "
            "Update DATA_DIR or the file name in the configuration cell."
        )
    dataframe = pd.read_csv(path)
    print(f"{dataset_name}: {dataframe.shape}")
    return dataframe


def remove_non_model_columns(dataframe: pd.DataFrame, columns_to_remove: list[str]) -> pd.DataFrame:
    """Remove columns that should not be used during model training."""
    return dataframe.drop(columns=columns_to_remove, errors="ignore").copy()


def encode_categorical_features(
    dataframes: list[pd.DataFrame],
    categorical_columns: list[str],
) -> list[pd.DataFrame]:
    """Encode categorical columns independently for each dataset.

    This keeps the notebook close to the original experiment, while keeping the code organized.
    """
    encoded_dataframes = []
    for dataframe in dataframes:
        dataframe = dataframe.copy()
        for column in categorical_columns:
            if column in dataframe.columns:
                dataframe[column] = LabelEncoder().fit_transform(dataframe[column].astype(str))
        encoded_dataframes.append(dataframe)
    return encoded_dataframes


def standardize_numeric_types(dataframe: pd.DataFrame, label_column: str) -> pd.DataFrame:
    """Convert numeric features to float while preserving the label column as integer."""
    dataframe = dataframe.copy()
    for column in dataframe.columns:
        if column == label_column:
            dataframe[column] = dataframe[column].astype(int)
        elif pd.api.types.is_numeric_dtype(dataframe[column]):
            dataframe[column] = dataframe[column].astype(float)
    return dataframe


def print_dataset_summary(dataframe: pd.DataFrame, dataset_name: str, label_column: str = LABEL_COLUMN) -> None:
    """Print dataset shape and class distribution."""
    print(f"\n{dataset_name}")
    print(f"Shape: {dataframe.shape}")
    if label_column in dataframe.columns:
        print("Class counts:")
        print(dataframe[label_column].value_counts().sort_index())
        print("Class distribution:")
        print(dataframe[label_column].value_counts(normalize=True).sort_index().map("{:.2%}".format))


def select_class_subset(
    dataframe: pd.DataFrame,
    class_value: int,
    integration_rate: float,
    label_column: str = LABEL_COLUMN,
    sort_column: str | None = TIME_COLUMN,
) -> pd.DataFrame:
    """Select a percentage of flows from a specific class."""
    class_flows = dataframe[dataframe[label_column] == class_value].copy()
    if sort_column in class_flows.columns:
        class_flows = class_flows.sort_values(by=sort_column)
    subset_size = int(len(class_flows) * integration_rate)
    return class_flows.iloc[:subset_size].copy()


def integrate_class_flows(
    training_dataframe: pd.DataFrame,
    external_dataframe: pd.DataFrame,
    class_value: int,
    integration_rate: float,
    training_name: str,
    external_name: str,
    class_name: str,
    label_column: str = LABEL_COLUMN,
    time_column: str = TIME_COLUMN,
    source_column: str = SOURCE_COLUMN,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Integrate class-specific external flows into the training dataset without data leakage.

    Returns:
        integrated_training: training dataset after removal and integration.
        external_test: external dataset after removing integrated flows.
        integrated_external_flows: external flows added to the training dataset.
        removed_training_flows: training flows removed to preserve dataset size.
    """
    integrated_external_flows = select_class_subset(
        external_dataframe,
        class_value=class_value,
        integration_rate=integration_rate,
        label_column=label_column,
        sort_column=time_column,
    )

    external_test = external_dataframe.drop(index=integrated_external_flows.index).copy()

    training_class_flows = training_dataframe[training_dataframe[label_column] == class_value].copy()
    if time_column in training_class_flows.columns:
        training_class_flows = training_class_flows.sort_values(by=time_column)

    removal_size = min(len(integrated_external_flows), len(training_class_flows))
    removed_training_flows = training_class_flows.iloc[:removal_size].copy()
    training_after_removal = training_dataframe.drop(index=removed_training_flows.index).copy()

    training_after_removal[source_column] = training_name.lower()
    integrated_external_flows[source_column] = (
        f"{external_name.lower()}_{int(integration_rate * 100)}_{class_name.lower()}"
    )

    integrated_training = pd.concat(
        [training_after_removal, integrated_external_flows],
        ignore_index=True,
    )

    if time_column in integrated_training.columns:
        integrated_training = integrated_training.sort_values(by=time_column).reset_index(drop=True)

    return integrated_training, external_test, integrated_external_flows, removed_training_flows


def split_scale_and_apply_pca(
    training_dataframe: pd.DataFrame,
    external_test_dataframe: pd.DataFrame,
    label_column: str = LABEL_COLUMN,
    source_column: str = SOURCE_COLUMN,
    test_size: float = TEST_SIZE,
    pca_components: int = PCA_COMPONENTS,
    random_state: int = RANDOM_STATE,
):
    """Split data, scale features, and apply PCA."""
    training_dataframe = training_dataframe.drop(columns=[source_column], errors="ignore").copy()
    external_test_dataframe = external_test_dataframe.drop(columns=[source_column], errors="ignore").copy()

    X = training_dataframe.drop(columns=[label_column])
    y = training_dataframe[label_column]

    X_external = external_test_dataframe.drop(columns=[label_column])
    y_external = external_test_dataframe[label_column]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y,
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    X_external_scaled = scaler.transform(X_external)

    pca = PCA(n_components=pca_components, random_state=random_state)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)
    X_external_pca = pca.transform(X_external_scaled)

    return (
        X_train_pca,
        X_test_pca,
        X_external_pca,
        y_train,
        y_test,
        y_external,
        X.columns,
        scaler,
        pca,
    )


def build_xgboost_model(num_classes: int, random_state: int = RANDOM_STATE) -> XGBClassifier:
    """Create the XGBoost classifier used in the experiment."""
    return XGBClassifier(
        eval_metric="mlogloss",
        n_estimators=300,
        max_depth=10,
        objective="multi:softprob",
        num_class=num_classes,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=0.5,
        random_state=random_state,
    )


def compute_false_alarm_rate(confusion: np.ndarray, class_labels: list[int]) -> dict[int, float]:
    """Compute the false alarm rate for each class from a confusion matrix."""
    far_per_class = {}
    for index, class_label in enumerate(class_labels):
        tp = confusion[index, index]
        fp = confusion[:, index].sum() - tp
        fn = confusion[index, :].sum() - tp
        tn = confusion.sum() - (tp + fp + fn)
        far_per_class[class_label] = fp / (fp + tn) if (fp + tn) > 0 else np.nan
    return far_per_class


def evaluate_model(model, X, y, dataset_name: str, class_labels: list[int]) -> dict:
    """Evaluate the model and print standard IDS metrics."""
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)
    confusion = confusion_matrix(y, y_pred, labels=class_labels)
    far_per_class = compute_false_alarm_rate(confusion, class_labels)

    metrics = {
        "dataset": dataset_name,
        "accuracy": accuracy_score(y, y_pred),
        "precision_macro": precision_score(y, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y, y_pred, average="macro", zero_division=0),
    }

    try:
        y_bin = label_binarize(y, classes=class_labels)
        metrics["auc_roc_ovr"] = roc_auc_score(y_bin, y_proba, average="macro", multi_class="ovr")
        metrics["auc_pr_macro"] = average_precision_score(y_bin, y_proba, average="macro")
    except ValueError:
        metrics["auc_roc_ovr"] = np.nan
        metrics["auc_pr_macro"] = np.nan

    for class_label, far in far_per_class.items():
        metrics[f"far_class_{class_label}"] = far

    print(f"\nEvaluation: {dataset_name}")
    print("Confusion Matrix:")
    print(confusion)
    print("\nClassification Report:")
    print(classification_report(y, y_pred, digits=4, zero_division=0))
    print("Metrics:")
    for key, value in metrics.items():
        if key != "dataset":
            print(f"{key}: {value:.4f}" if pd.notna(value) else f"{key}: NaN")

    return metrics


def plot_confusion_matrix(model, X, y, dataset_name: str, class_labels: list[int]) -> None:
    """Plot the confusion matrix."""
    ConfusionMatrixDisplay.from_estimator(
        model,
        X,
        y,
        labels=class_labels,
        values_format="d",
    )
    plt.title(f"Confusion Matrix — {dataset_name}")
    plt.tight_layout()
    plt.show()


def plot_multiclass_roc_curve(model, X, y, dataset_name: str, class_labels: list[int]) -> None:
    """Plot one-vs-rest ROC curves for a multiclass classifier."""
    y_bin = label_binarize(y, classes=class_labels)
    y_proba = model.predict_proba(X)

    plt.figure(figsize=(8, 6))
    for index, class_label in enumerate(class_labels):
        fpr, tpr, _ = roc_curve(y_bin[:, index], y_proba[:, index])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, lw=2, label=f"Class {class_label} (AUC = {roc_auc:.4f})")

    plt.plot([0, 1], [0, 1], linestyle="--", lw=1)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve — {dataset_name}")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.show()

## 4. Load Datasets

In [ ]:
df_cic17_raw = load_dataset(CIC17_PATH, "GenIDS-CIC17")
df_cic18_raw = load_dataset(CIC18_PATH, "GenIDS-CIC18")

## 5. Preprocessing

Non-predictive identifiers and dataset-specific metadata columns are removed. Categorical features are encoded and numeric types are standardized.

In [ ]:
cic17_columns_to_remove = [
    "binary",
    "date",
    "hours",
    "expiration_id",
    "src_ip",
    "src_mac",
    "src_oui",
    "dst_ip",
    "dst_mac",
    "dst_oui",
    "ip_version",
    "vlan_id",
    "tunnel_id",
]

cic18_columns_to_remove = [
    "binary",
    "src_ip",
    "src_mac",
    "src_oui",
    "dst_ip",
    "dst_mac",
    "dst_oui",
    "ip_version",
    "vlan_id",
    "tunnel_id",
]

df_cic17 = remove_non_model_columns(df_cic17_raw, cic17_columns_to_remove)
df_cic18 = remove_non_model_columns(df_cic18_raw, cic18_columns_to_remove)

categorical_feature_columns = ["application_name", "application_category_name", LABEL_COLUMN]
df_cic17, df_cic18 = encode_categorical_features(
    [df_cic17, df_cic18],
    categorical_feature_columns,
)

df_cic17 = standardize_numeric_types(df_cic17, LABEL_COLUMN)
df_cic18 = standardize_numeric_types(df_cic18, LABEL_COLUMN)

print_dataset_summary(df_cic17, "GenIDS-CIC17")
print_dataset_summary(df_cic18, "GenIDS-CIC18")

## 6. Malicious (D)DoS Flow Integration

Malicious **(D)DoS** flows from CIC17 are integrated into CIC18. The selected CIC17 flows are removed from the CIC17 external evaluation dataset to avoid data leakage. An equivalent number of CIC18 **(D)DoS** flows is removed from CIC18 before integration to keep the training set size consistent.

In [ ]:
integrated_cic18, cic17_test, integrated_cic17_ddos, removed_cic18_ddos = integrate_class_flows(
    training_dataframe=df_cic18,
    external_dataframe=df_cic17,
    class_value=DDOS_CLASS_VALUE,
    integration_rate=INTEGRATION_RATE,
    training_name="CIC18",
    external_name="CIC17",
    class_name="ddos",
)

print_dataset_summary(integrated_cic18, "Integrated GenIDS-CIC18")
print_dataset_summary(cic17_test, "GenIDS-CIC17 after leakage-safe removal")

print(f"\nIntegrated CIC17 (D)DoS flows: {integrated_cic17_ddos.shape}")
print(f"Removed CIC18 (D)DoS flows: {removed_cic18_ddos.shape}")

## 7. Optional Visualization of Integrated Flows

In [ ]:
if TIME_COLUMN in integrated_cic18.columns and "bidirectional_bytes" in integrated_cic18.columns:
    plt.figure(figsize=(16, 8))

    training_flows = integrated_cic18[integrated_cic18[SOURCE_COLUMN] == "cic18"]
    integrated_flows = integrated_cic18[
        integrated_cic18[SOURCE_COLUMN] == f"cic17_{int(INTEGRATION_RATE * 100)}_ddos"
    ]

    plt.scatter(
        training_flows[TIME_COLUMN],
        training_flows["bidirectional_bytes"],
        label="CIC18 flows",
        alpha=0.5,
    )
    plt.scatter(
        integrated_flows[TIME_COLUMN],
        integrated_flows["bidirectional_bytes"],
        label="Integrated (D)DoS flows from CIC17",
        marker="x",
        alpha=0.8,
    )

    plt.title("Flow Volume over Time after Malicious Flow Integration")
    plt.xlabel("Timestamp (bidirectional_first_seen_ms)")
    plt.ylabel("Data Volume (bidirectional_bytes)")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.show()
else:
    print("The required columns for visualization are not available.")

## 8. Train-Test Split, Normalization, and PCA

The model is trained using the integrated CIC18 dataset. CIC17 is used as the external cross-dataset generalization test. Standardization is applied before PCA.

In [ ]:
(
    X_train_pca,
    X_test_pca,
    X_cic17_pca,
    y_train,
    y_test,
    y_cic17,
    feature_names,
    scaler,
    pca,
) = split_scale_and_apply_pca(
    training_dataframe=integrated_cic18,
    external_test_dataframe=cic17_test,
)

print(f"Training set after PCA: {X_train_pca.shape}")
print(f"Internal test set after PCA: {X_test_pca.shape}")
print(f"External test set after PCA: {X_cic17_pca.shape}")
print(f"Cumulative explained variance: {pca.explained_variance_ratio_.sum():.4f}")

## 9. PCA Analysis

In [ ]:
explained_variance = pd.DataFrame({
    "principal_component": [f"PC{i + 1}" for i in range(pca.n_components_)],
    "explained_variance_ratio": pca.explained_variance_ratio_,
    "cumulative_explained_variance": np.cumsum(pca.explained_variance_ratio_),
})

explained_variance

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(
    explained_variance["principal_component"],
    explained_variance["cumulative_explained_variance"],
    marker="o",
)
plt.xticks(rotation=90)
plt.xlabel("Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("Cumulative Explained Variance by PCA Components")
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
loadings = pd.DataFrame(
    data=pca.components_,
    columns=feature_names,
    index=[f"PC{i + 1}" for i in range(pca.n_components_)],
)

top_k = 5
top_features_by_pc = {
    pc: loadings.loc[pc].abs().sort_values(ascending=False).head(top_k)
    for pc in loadings.index
}

for pc, features in top_features_by_pc.items():
    print(f"\n{pc} — Top {top_k} features:")
    print(features)

## 10. Model Training

In [ ]:
class_labels = sorted(pd.concat([y_train, y_test, y_cic17]).unique().tolist())
num_classes = len(class_labels)

model = build_xgboost_model(num_classes=num_classes, random_state=RANDOM_STATE)
model.fit(X_train_pca, y_train)

print("Model training completed.")
print(f"Class labels: {class_labels}")

## 11. Principal Component Importance

In [ ]:
pca_importance = pd.DataFrame({
    "principal_component": [f"PC{i + 1}" for i in range(len(model.feature_importances_))],
    "importance": model.feature_importances_,
}).sort_values(by="importance", ascending=False)

pca_importance.head(20)

In [ ]:
top_10_components = pca_importance.head(10).sort_values(by="importance")

plt.figure(figsize=(10, 6))
plt.barh(top_10_components["principal_component"], top_10_components["importance"])
plt.title("Top 10 Principal Components by XGBoost Importance")
plt.xlabel("Importance")
plt.ylabel("Principal Component")
plt.tight_layout()
plt.show()

## 12. Internal Evaluation — Integrated CIC18

In [ ]:
internal_metrics = evaluate_model(
    model=model,
    X=X_test_pca,
    y=y_test,
    dataset_name="Integrated CIC18 internal test",
    class_labels=class_labels,
)

plot_confusion_matrix(
    model=model,
    X=X_test_pca,
    y=y_test,
    dataset_name="Integrated CIC18 internal test",
    class_labels=class_labels,
)

plot_multiclass_roc_curve(
    model=model,
    X=X_test_pca,
    y=y_test,
    dataset_name="Integrated CIC18 internal test",
    class_labels=class_labels,
)

## 13. Cross-Dataset Generalization Evaluation — CIC17

In [ ]:
external_metrics = evaluate_model(
    model=model,
    X=X_cic17_pca,
    y=y_cic17,
    dataset_name="CIC17 generalization test",
    class_labels=class_labels,
)

plot_confusion_matrix(
    model=model,
    X=X_cic17_pca,
    y=y_cic17,
    dataset_name="CIC17 generalization test",
    class_labels=class_labels,
)

plot_multiclass_roc_curve(
    model=model,
    X=X_cic17_pca,
    y=y_cic17,
    dataset_name="CIC17 generalization test",
    class_labels=class_labels,
)

## 14. Save Metrics

In [ ]:
metrics = pd.DataFrame([internal_metrics, external_metrics])
metrics_path = (
    RESULTS_DIR
    / f"pca_malicious_flow_integration_{int(INTEGRATION_RATE * 100)}pct_cic17_to_cic18_metrics.csv"
)
metrics.to_csv(metrics_path, index=False)

print(f"Metrics saved to: {metrics_path}")
metrics